# 4chan Scraper Notebook

In [ ]:
import json
from pathlib import Path

from chud.scraper import ScraperChan, Post
from chud.preprocess import DataProcessor, save_posts, load_posts

## Scraping

In [ ]:
BOARDS = ["g"]
MAX_THREADS_PER_BOARD = 50
MIN_REPLIES = 5
OUTPUT_FILE = "./data/posts.json"

In [ ]:
scraper = ScraperChan()
all_posts = []

for board in BOARDS:
    print(f"\n{'='*50}")
    print(f'Scraping /{board}/...')
    print(f"{'='*50}")
    
    posts = scraper.scrape_board(
        board,
        max_threads=MAX_THREADS_PER_BOARD,
        min_replies=MIN_REPLIES,
        verbose=True
    )
    
    all_posts.extend(posts)
    print(f'\nScraped {len(posts)} posts from /{board}/')

print(f"\n{'='*50}")
print(f'TOTAL POSTS SCRAPED: {len(all_posts)}')
print(f"{'='*50}")

In [ ]:
import statistics

comment_lengths = [len(p.comment) for p in all_posts]
op_posts = sum(1 for p in all_posts if p.is_op)
reply_posts = sum(1 for p in all_posts if p.replies_to)

print("Dataset Statistics:")
print(f"  Total posts: {len(all_posts)}")
print(f"  OP posts: {op_posts}")
print(f"  Posts with replies: {reply_posts}")
print(f"  Avg comment length: {statistics.mean(comment_lengths):.1f} chars")
print(f"  Median comment length: {statistics.median(comment_lengths):.1f} chars")
print(f"  Min/Max length: {min(comment_lengths)} / {max(comment_lengths)} chars")

## Process and Filter Data

In [ ]:
MIN_POST_LENGTH = 20
MAX_POST_LENGTH = 2000
MAX_QUOTE_RATIO = 0.7

processor = DataProcessor(
    min_length=MIN_POST_LENGTH,
    max_length=MAX_POST_LENGTH,
    max_quote_ratio=MAX_QUOTE_RATIO
)

In [ ]:
completion_data = processor.build_completion_data(all_posts)
print(f"Completion examples: {len(completion_data)}")

conversation_pairs = processor.build_conversation_pairs(all_posts)
print(f"Conversation pairs: {len(conversation_pairs)}")

In [ ]:
print("Sample conversation pairs:\n")

for i, pair in enumerate(conversation_pairs[:3]):
    print(f"--- Pair {i+1} ---")
    print(f"INPUT: {pair['input'][:200]}..." if len(pair['input']) > 200 else f"INPUT: {pair['input']}")
    print()
    print(f"OUTPUT: {pair['output'][:200]}..." if len(pair['output']) > 200 else f"OUTPUT: {pair['output']}")
    print()

## Save Data

Save the scraped posts for later use in training:

In [ ]:
output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)

save_posts(all_posts, OUTPUT_FILE)
print(f"\nSaved {len(all_posts)} posts to {OUTPUT_FILE}")

In [ ]:
completion_file = output_path.parent / "completion_data.json"
conversation_file = output_path.parent / "conversation_data.json"

with open(completion_file, 'w') as f:
    json.dump(completion_data, f, indent=2)
print(f"Saved completion data to {completion_file}")

with open(conversation_file, 'w') as f:
    json.dump(conversation_pairs, f, indent=2)
print(f"Saved conversation data to {conversation_file}")